# Lecture 3 · From a question to a grounded answer

> How many books can a student borrow at a time, for how long, and how can they renew them?

A student asks one question whose answer spans the borrowing table on page 1 and the renewal instructions on page 2. We will send that question to the index, follow the returned passages back to the guide, and check what the answer model actually receives.

**Before you run this lecture**, finish [Lecture 2 · Build the index](lecture_2_pdf_to_index.ipynb). Start a fresh kernel in the project root or `lectures/` using the project’s Jupyter environment. Keep the same `lectures/.env`, embedding model and vector dimensions used for indexing. Live queries use the configured model APIs. The original PDF and extracted diagram are useful for checking the answer and running the image experiment. [Library Borrowing Guide](../data/library-guide.pdf) · [Environment setup](../README.md)

| Inspect | What to look for |
| --- | --- |
| [Retrieved evidence](#step-7) | Did the search find both the borrowing table and the reservation condition? |
| [Context and prompt](#step-8) | Where do the passages, instructions and original question appear? |
| [Answer and cache](#step-9) | Which claims do the sources support? Did a new answer-model call occur? |


## Open the index from Lecture 2

The index lives on disk. Only the model callbacks and the connection to storage need to be recreated in this kernel. The callbacks record model activity locally so you can inspect whether the answer came from a new call and whether an image was sent in the final experiment.


In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from lightrag import QueryParam
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc
from raganything import RAGAnything, RAGAnythingConfig, set_prompt_language

project_dir = Path.cwd()
if project_dir.name == "lectures":
    project_dir = project_dir.parent
parsed_dir = project_dir / "outputs/parsed"
storage_dir = project_dir / "outputs/library_rag_storage"

status_path = storage_dir / "kv_store_doc_status.json"
if not status_path.is_file():
    raise FileNotFoundError(f"Run Lecture 2 first to create the index at {storage_dir}.")
documents = json.loads(status_path.read_text(encoding="utf-8"))
library_doc_ids = [
    doc_id for doc_id, doc in documents.items()
    if Path(doc.get("file_path") or "").name == "library-guide.pdf"
    and doc.get("status") == "processed"
]
if not library_doc_ids:
    raise RuntimeError("Finish indexing library-guide.pdf in Lecture 2, then run this cell again.")

load_dotenv(project_dir / "lectures/.env")
model_settings = ("OPENAI_API_KEY", "LLM_MODEL", "VISION_MODEL", "EMBEDDING_MODEL", "EMBEDDING_DIM")
missing = [name for name in model_settings if not os.getenv(name)]
if missing:
    raise ValueError(f"Set {', '.join(missing)} in lectures/.env (see lectures/.env.example) before running live queries.")
api_key = os.environ["OPENAI_API_KEY"]
base_url = os.getenv("OPENAI_BASE_URL")
llm_model = os.environ["LLM_MODEL"]
vision_model = os.environ["VISION_MODEL"]
embedding_model = os.environ["EMBEDDING_MODEL"]
embedding_dim = int(os.environ["EMBEDDING_DIM"])
print(f"""Index: {storage_dir}
Query embedding model: {embedding_model} ({embedding_dim} dimensions)""")


Index: /Users/weiningxu/Documents/Project/rag_anything_playground/outputs/library_rag_storage
Query embedding model: text-embedding-3-small (1536 dimensions)


In [2]:
# Keep model inputs and outputs so we can read them during the core steps.
calls = []

async def llm(prompt, system_prompt=None, history_messages=None, **kwargs):
    output = await openai_complete_if_cache(
        llm_model, prompt, system_prompt=system_prompt,
        history_messages=history_messages, base_url=base_url, api_key=api_key, **kwargs,
    )
    calls.append({
        "kind": "llm", "prompt": prompt, "system_prompt": system_prompt,
        "history_messages": history_messages or [], "output": output,
    })
    return output

async def vision(prompt, system_prompt=None, image_data=None, messages=None, **kwargs):
    # Ingestion supplies a PNG; image-enhanced queries supply complete messages.
    if messages is None:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        content = [{"type": "text", "text": prompt}]
        if image_data:
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_data}"},
            })
        messages.append({"role": "user", "content": content})

    output = await openai_complete_if_cache(
        vision_model, "", messages=messages, base_url=base_url, api_key=api_key, **kwargs,
    )
    image_count = sum(
        part["type"] == "image_url"
        for message in messages if isinstance(message["content"], list)
        for part in message["content"]
    )
    calls.append({
        "kind": "vision", "prompt": prompt, "system_prompt": system_prompt,
        "images": image_count, "output": output,
    })
    return output

async def embed(texts):
    vectors = await openai_embed.func(
        texts, model=embedding_model, base_url=base_url, api_key=api_key,
        embedding_dim=embedding_dim, max_token_size=8192,
    )
    calls.append({
        "kind": "embed", "texts": list(texts),
        "shape": vectors.shape, "sample": vectors[0, :6].tolist(),
    })
    return vectors


In [3]:
set_prompt_language("en")

config = RAGAnythingConfig(
    parser="docling",
    parser_output_dir=str(parsed_dir),
    working_dir=str(storage_dir),
    display_content_stats=False,
)

rag = RAGAnything(
    config=config,
    llm_model_func=llm,
    vision_model_func=vision,
    embedding_func=EmbeddingFunc(
        embedding_dim=embedding_dim,
        max_token_size=8192,
        model_name=embedding_model,
        func=embed,
    ),
    lightrag_kwargs={
        "llm_model_name": llm_model,
        "chunk_token_size": 75,
        "chunk_overlap_token_size": 20,
        "addon_params": {"language": "English"},
    },
)


# RAGAnything creates its LightRAG engine lazily. Open it before the structured lookup.
opened = await rag._ensure_lightrag_initialized()
if not opened["success"]:
    raise RuntimeError(opened["error"])
for doc_id in library_doc_ids:
    status = await rag.get_document_processing_status(doc_id)
    if status["fully_processed"] and status["chunks_count"] > 0:
        print(f"Opened library-guide.pdf with {status['chunks_count']} indexed chunks.")
        break
else:
    raise RuntimeError("Finish the text, table and image indexing in Lecture 2 before querying.")


INFO: RAGAnything initialized with config:
INFO:   Working directory: /Users/weiningxu/Documents/Project/rag_anything_playground/outputs/library_rag_storage
INFO:   Parser: docling
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1
INFO: Parser 'docling' installation verified
INFO: Initializing LightRAG with parameters: {'working_dir': '/Users/weiningxu/Documents/Project/rag_anything_playground/outputs/library_rag_storage', 'llm_model_name': 'gpt-6-luna', 'chunk_token_size': 75, 'chunk_overlap_token_size': 20, 'addon_params': {'language': 'English'}}
INFO: [] Loaded graph from /Users/weiningxu/Documents/Project/rag_anything_playground/outputs/library_rag_storage/graph_chunk_entity_relation.graphml with 33 nodes, 75 edges
INFO:nano-vectordb:Load (33, 1536) data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': '/Users/weiningxu/Documents/Project/rag_anything_playground/outp

Opened library-guide.pdf with 6 indexed chunks.


<a id="step-7"></a>
## Retrieve the evidence

The full question searches chunk vectors by meaning. In `mix` mode LightRAG also extracts or reuses **low-level keywords** such as *loan duration* to search entity vectors, and **high-level keywords** such as *book renewal process* to search relationship vectors. Matched nodes bring in connected edges. Matched edges bring in their endpoint nodes. Graph records carry `source_id` links to supporting chunks.

```mermaid
flowchart TD
    Q["Borrowing question"] --> CV["Embed full question → search chunk vectors"]
    Q --> K["Extract or reuse keywords"]
    K --> EN["Specific keywords → entity vectors → matching nodes and their edges"]
    K --> RE["Topic keywords → relationship vectors → matching edges and their nodes"]
    EN --> G["Merge graph facts within entity and relationship token limits"]
    RE --> G
    G --> SC["Follow source IDs to supporting chunks"]
    CV --> M["Merge candidate lists and deduplicate chunks"]
    SC --> M
    M --> C["Apply chunk count and remaining context budget"]
    C --> D["Return chunks, graph facts and source references"]
```

`top_k` and the vector similarity threshold limit graph candidates. `max_entity_tokens` and `max_relation_tokens` bound the graph descriptions. `kg_chunk_pick_method` and `related_chunk_number` affect which linked chunks enter the merge. A configured reranker can rescore passages after the merge. Here reranking is disabled. `chunk_top_k` and the remaining `max_total_tokens` budget bound the final passages.

**Predict before running.** Which passage could answer “for how long” yet leave “how can they renew” unresolved? Read the actual keywords, result counts, and first graph records below. The structured lookup can call the keyword and embedding models and returns evidence for inspection.


In [4]:
question = "How many books can a student borrow at a time, for how long, and how can they renew them?"
retrieval = await rag.lightrag.aquery_data(
    question, QueryParam(mode="mix", enable_rerank=False)
)
print(f"""{question}
Retrieval: {retrieval['status']} {retrieval['message']}""")
if retrieval["status"] != "success":
    raise RuntimeError(f"Retrieval could not supply evidence: {retrieval['message']}")

data = retrieval["data"]
entities = data.get("entities", [])
relationships = data.get("relationships", [])
retrieved_chunks = data.get("chunks", [])
references = data.get("references", [])
keywords = retrieval.get("metadata", {}).get("keywords", {})
example_entity = next(iter(entities), None)
example_relationship = next(iter(relationships), None)

print(f"""Specific keywords: {keywords.get('low_level', [])}
Topic keywords: {keywords.get('high_level', [])}
entities {len(entities)}
relationships {len(relationships)}
chunks {len(retrieved_chunks)}
references {len(references)}
Example entity: {example_entity}
Example relationship: {example_relationship}
Source files: {references}""")


INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: Query nodes: Books, Student, Number of books borrowed, Borrowing duration, Renewal process (top_k:40, cosine:0.2)
INFO: Local query: 33 entites, 75 relations
INFO: Query edges: Student library borrowing policy, Loan period, Book renewal (top_k:40, cosine:0.2)
INFO: Global query: 23 entites, 40 relations
INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 33 entities, 75 relations, 6 vector chunks
INFO: After truncation: 33 entities, 75 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 75 relations
INFO: Round-robin merged chunks: 12 -> 6 (deduplicated 6)
INFO: Final context: 33 entities, 75 relations, 6 chunks
INFO: Final chunks S+F/O: E8/1 E18/2 E10/3 E11/4 E11/5 E1/6


How many books can a student borrow at a time, for how long, and how can they renew them?
Retrieval: success Query processed successfully
Specific keywords: ['Books', 'Student', 'Number of books borrowed', 'Borrowing duration', 'Renewal process']
Topic keywords: ['Student library borrowing policy', 'Loan period', 'Book renewal']
entities 33
relationships 75
chunks 6
references 1
Example entity: {'entity_name': 'Student book borrowing limits and standard loan period (table)', 'entity_type': 'table', 'description': 'The table states that students may borrow a maximum of 3 books at a time and that the standard loan period is 14 days. In the Library Borrowing Guide, these rules frame instructions to check eligibility, track each book’s due date in My Loans, and return books on time. A separate conditional renewal rule allows one 7-day extension when no reservation prevents it.', 'source_id': 'chunk-73e03336d64bd8478972d0d7b5bf162b', 'file_path': 'library-guide.pdf', 'created_at': 179505749

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


<a id="source-trace"></a>
### Read the evidence and trace a claim to its page

The saved structured-query output in the [core workflow notebook](rag_anything_core_workflow.ipynb#step-7) showed 30 entities, 65 relationships, 6 chunks and 1 file in one run. Entities and edges are navigation aids. To check whether a student can renew a reserved book, read the words in a retrieved chunk.

**Worked trace from the same historical run**

1. The retrieved entity `Student Book Borrowing Limits and Standard Loan Period (table)` described the 3-book limit and 14-day period. Its `source_id`, `chunk-604374813fb8c7f3a8154e53d151ad0b`, pointed back to the table chunk from page 1.
2. The structured query also returned `chunk-f74d3981177947bc583a0860d03ffe3f`. Its page range covers pages 1–2. The following excerpt comes from the page 2 portion.

   > Open My Loans, find the book, and click "Renew".
   >
   > If nobody else has reserved it, renew once for 7 extra days.

3. The returned chunk had `reference_id="1"`. The references list mapped `1` to `library-guide.pdf`. That mapping explains the `[1]` beside the renewal sentence in the [historical answer below](#historical-answer).

The cell below looks **only inside the passages the search returned**. It prints an excerpt around each expected phrase, its chunk ID, its file reference, and the page range stored with that chunk. The phrases serve as reading aids for this guide. A missing literal calls for inspecting the returned text for equivalent wording. `page_idx` starts at zero. A chunk spanning pages 1–2 can contain the renewal condition from page 2. Compare each excerpt with the [source PDF](../data/library-guide.pdf).


In [5]:
for phrase in ("3 books", "14 days", "If nobody else has reserved"):
    match_text = "No literal match in the retrieved passages."
    for chunk in retrieved_chunks:
        content = chunk["content"]
        position = content.casefold().find(phrase.casefold())
        if position < 0:
            continue
        stored = await rag.lightrag.text_chunks.get_by_id(chunk["chunk_id"])
        first = (stored or {}).get("page_idx")
        last = (stored or {}).get("page_idx_end")
        page_label = "page unavailable"
        if first is not None:
            page_label = f"page {first + 1}"
            if last is not None and last != first:
                page_label = f"pages {first + 1}–{last + 1}"
        source_label = f"{chunk['chunk_id']} | [{chunk['reference_id']}] {chunk['file_path']} | PDF {page_label}"
        excerpt = content[max(0, position - 60):position + len(phrase) + 100].strip()
        match_text = f"{source_label}\n{excerpt}"
        break

    print(f"""
Looking for {phrase!r}
{match_text}""")

# Change the index to read another retrieved chunk in full.
retrieved_chunk_index = 0
if retrieved_chunks:
    print("\nFull passage:", retrieved_chunks[retrieved_chunk_index]["content"])



Looking for '3 books'
chunk-73e03336d64bd8478972d0d7b5bf162b | [1] library-guide.pdf | PDF page 1
start_col_offset_idx': 1, 'end_col_offset_idx': 2, 'text': '3 books', 'column_header': False, 'row_header': False, 'row_section': False, 'fillable': False}, {'bbox': {

Looking for '14 days'
chunk-73e03336d64bd8478972d0d7b5bf162b | [1] library-guide.pdf | PDF page 1
start_col_offset_idx': 1, 'end_col_offset_idx': 2, 'text': '14 days', 'column_header': False, 'row_header': False, 'row_section': False, 'fillable': False}], 'num_rows

Looking for 'If nobody else has reserved'
chunk-f74d3981177947bc583a0860d03ffe3f | [1] library-guide.pdf | PDF pages 1–2
ue date.

Open My Loans, find the book, and click "Renew".

If nobody else has reserved it, renew once for 7 extra days.

If it is reserved

Full passage: check each title separately.

Return each book on or before its due date.

Library Borrowing Guide

1 / 2

How to Renew

If you need more time, request a renewal before the due date.

Open 

`reference_id` identifies a file in the result. `chunk_id` identifies one passage. When two retrieved chunks both cite `[1]`, they may support different clauses from different pages of the same guide. To check a graph claim, follow its own `source_id` to its supporting chunks.

### Experiment 1 · Compare the same question in two retrieval modes

| Mode | Entry point |
| --- | --- |
| `naive` | Search chunk vectors directly |
| `local` | Search entity vectors, then connected graph facts and source chunks |
| `global` | Search relationship vectors, then endpoint nodes and source chunks |
| `hybrid` | Combine the local and global graph routes |
| `mix` | Add direct chunk-vector search to the hybrid route |
| `bypass` | Pass the request to the model without retrieval |

The earlier run returned six chunks in both `mix` and `naive`. Only `mix` returned graph facts. Check your results and ask whether each mode returned the reservation condition. Set `run_experiments` to `False` to skip both optional comparisons, including the diagram experiment near the end.


In [6]:
run_experiments = True

naive = None
if run_experiments:
    naive = await rag.lightrag.aquery_data(question, QueryParam(mode="naive", enable_rerank=False))

if naive is not None:
    for mode, result in (("mix", retrieval), ("naive", naive)):
        mode_data = result["data"]
        chunks = mode_data.get("chunks", [])
        condition = "if nobody else has reserved"
        hits = [
            chunk["chunk_id"] for chunk in chunks
            if condition in chunk["content"].casefold()
        ]
        print(f"""{mode}
  chunks {len(chunks)}
  entities {len(mode_data.get('entities', []))}
  relationships {len(mode_data.get('relationships', []))}
  Passages containing {condition!r}: {hits}""")
else:
    print("Experiment skipped.")


INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 6 chunks


mix
  chunks 6
  entities 33
  relationships 75
  Passages containing 'if nobody else has reserved': ['chunk-f74d3981177947bc583a0860d03ffe3f', 'chunk-73e03336d64bd8478972d0d7b5bf162b', 'chunk-b5b66dc4cdc65aee472efcef7cbf0db9', 'chunk-72b56bc685fea8dbd9e08a18a4a7c1b1']
naive
  chunks 6
  entities 0
  relationships 0
  Passages containing 'if nobody else has reserved': ['chunk-f74d3981177947bc583a0860d03ffe3f', 'chunk-73e03336d64bd8478972d0d7b5bf162b', 'chunk-b5b66dc4cdc65aee472efcef7cbf0db9', 'chunk-72b56bc685fea8dbd9e08a18a4a7c1b1']


<a id="step-8"></a>
## Assemble the context and preview the prompt

Where does “7 extra days” go after retrieval? The **context** contains selected graph facts, readable passages and source labels. In the assembled **system message**, LightRAG combines instructions about using evidence with that context. The **user message** contains the original question.

| Observation | Return value |
| --- | --- |
| `aquery_data` above | Structured chunks, graph records and references |
| `only_need_context=True` | The assembled evidence as text |
| `only_need_prompt=True` | A preview of instructions, evidence and question |
| Ordinary `aquery` in the next section | The student-facing answer |

Each call performs its own retrieval. The first three omit final answer generation but can still use model APIs. An application that only needs the answer can make the fourth call alone.

**Inspect the context.** Find the loan limit in a table-derived passage and the renewal condition in a text passage.


In [7]:
context = await rag.aquery(
    question, mode="mix", only_need_context=True, vlm_enhanced=False, enable_rerank=False,
)

print(context)


INFO: Executing text query: How many books can a student borrow at a time, for how long, and how can they renew them?...
INFO: Query mode: mix
INFO: Query nodes: Books, Student, Number of books borrowed, Borrowing duration, Renewal process (top_k:40, cosine:0.2)
INFO: Local query: 33 entites, 75 relations
INFO: Query edges: Student library borrowing policy, Loan period, Book renewal (top_k:40, cosine:0.2)
INFO: Global query: 23 entites, 40 relations
INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 33 entities, 75 relations, 6 vector chunks
INFO: After truncation: 33 entities, 75 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 75 relations
INFO: Round-robin merged chunks: 12 -> 6 (deduplicated 6)
INFO: Final context: 33 entities, 75 relations, 6 chunks
INFO: Final chunks S+F/O: E8/1 E18/2 E10/3 E11/4 E11/5 E1/6
INFO: Text query completed



Knowledge Graph Data (Entity):

```json
{"entity": "Student book borrowing limits and standard loan period (table)", "type": "table", "description": "The table states that students may borrow a maximum of 3 books at a time and that the standard loan period is 14 days. In the Library Borrowing Guide, these rules frame instructions to check eligibility, track each book’s due date in My Loans, and return books on time. A separate conditional renewal rule allows one 7-day extension when no reservation prevents it."}
{"entity": "Library Borrowing Guide", "type": "content", "description": "Library Borrowing Guide is the title of the displayed content.<SEP>The Library Borrowing Guide provides instructions for borrowing and returning books.<SEP>The Library Borrowing Guide provides instructions for renewing and returning a book.<SEP>The Library Borrowing Guide provides instructions to check titles separately, return books by their due dates, and request renewals.<SEP>The Library Borrowing Guid

**Inspect the prompt.** Find the instruction that constrains the answer to the supplied evidence, then find the student's question. The preview lets you inspect what has been prepared before asking for an answer.


In [8]:
prompt_preview = await rag.aquery(
    question, mode="mix", only_need_prompt=True, vlm_enhanced=False, enable_rerank=False,
)

print(prompt_preview)


INFO: Executing text query: How many books can a student borrow at a time, for how long, and how can they renew them?...
INFO: Query mode: mix
INFO: Query nodes: Books, Student, Number of books borrowed, Borrowing duration, Renewal process (top_k:40, cosine:0.2)
INFO: Local query: 33 entites, 75 relations
INFO: Query edges: Student library borrowing policy, Loan period, Book renewal (top_k:40, cosine:0.2)
INFO: Global query: 23 entites, 40 relations
INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 33 entities, 75 relations, 6 vector chunks
INFO: After truncation: 33 entities, 75 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 75 relations
INFO: Round-robin merged chunks: 12 -> 6 (deduplicated 6)
INFO: Final context: 33 entities, 75 relations, 6 chunks
INFO: Final chunks S+F/O: E8/1 E18/2 E10/3 E11/4 E11/5 E1/6
INFO: Text query completed


---Role---

You are an expert AI assistant specializing in synthesizing information from a provided knowledge base. Your primary function is to answer user queries accurately by ONLY using the information within the provided **Context**.

---Goal---

Generate a comprehensive, well-structured answer to the user query.
The answer must integrate relevant facts from the Knowledge Graph and Document Chunks found in the **Context**.
Consider the conversation history if provided to maintain conversational flow and avoid repeating information.

---Instructions---

1. Step-by-Step Instruction:
  - Carefully determine the user's query intent in the context of the conversation history to fully understand the user's information need.
  - Scrutinize both `Knowledge Graph Data` and `Document Chunks` in the **Context**. Identify and extract all pieces of information that are directly relevant to answering the user query.
  - Weave the extracted facts into a coherent and logical response. Your own kno

<a id="step-9"></a>
## Generate the answer and check its claims

Run the ordinary `mix` query with text answering and read the answer first. Then look at the callback log below it. A fresh answer callback exposes the actual system prompt, user message and conversation history. A repeat can return a previously computed answer.

| Claim to check in the answer | Where to verify it |
| --- | --- |
| Up to 3 books at a time for 14 days | Borrowing table on page 1 |
| Request renewal before the due date, open My Loans, find the book and select Renew | Renewal instructions on page 2 |
| One renewal for 7 extra days if nobody else has reserved the book | Renewal condition on page 2 |
| Check the new due date after renewing | Diagram and confirmation text on page 2 |

<a id="historical-answer"></a>
**Answer generated in the historical core workflow run**

> Students may borrow **up to 3 books at a time**. The standard loan period is **14 days**. [1]
>
> To renew a book, request the renewal **before its due date**: open **My Loans**, find the book, and click **Renew**. If nobody else has reserved it, it can be renewed **once for 7 extra days**. If it is reserved, return it by the original due date. After renewing, check the new due date in **My Loans**. [1]
>
> **References**
>
> - [1] library-guide.pdf

Read this answer alongside the two chunk IDs in the [worked trace](#source-trace). The citation after each paragraph points to the same guide, while the limits and the renewal instructions come from different parts of it.

A response that says “renew for seven days” has dropped two conditions. Locate the words that restrict **how often** and **when renewal is allowed** and follow the reference back to the guide.


In [9]:
before_answer = len(calls)
answer = await rag.aquery(
    question, mode="mix", stream=False, vlm_enhanced=False, enable_rerank=False,
)
after_answer = len(calls)

answer_calls = [
    call for call in calls[before_answer:after_answer]
    if call["kind"] == "llm" and call["prompt"] == question and "output" in call
]
actual = answer_calls[-1] if answer_calls else None

no_call = "No new answer-model call was recorded."
if actual:
    activity = "A new answer-model call was recorded."
    system_prompt = actual["system_prompt"]
    user_question = actual["prompt"]
    history = actual["history_messages"]
else:
    activity = "No new answer-model call was recorded; a cached answer may have been used."
    system_prompt = user_question = history = no_call

print(f"""
Answer:
{answer}

Answer-model activity:
{activity}

Actual system prompt:
{system_prompt}

Actual user question:
{user_question}

Actual conversation history:
{history}""")


INFO: Executing text query: How many books can a student borrow at a time, for how long, and how can they renew them?...
INFO: Query mode: mix
INFO: Query nodes: Books, Student, Number of books borrowed, Borrowing duration, Renewal process (top_k:40, cosine:0.2)
INFO: Local query: 33 entites, 75 relations
INFO: Query edges: Student library borrowing policy, Loan period, Book renewal (top_k:40, cosine:0.2)
INFO: Global query: 23 entites, 40 relations
INFO: Naive query: 6 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 33 entities, 75 relations, 6 vector chunks
INFO: After truncation: 33 entities, 75 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 75 relations
INFO: Round-robin merged chunks: 12 -> 6 (deduplicated 6)
INFO: Final context: 33 entities, 75 relations, 6 chunks
INFO: Final chunks S+F/O: E8/1 E18/2 E10/3 E11/4 E11/5 E1/6
INFO:  == LLM cache == Query cache hit, using cached respons


Answer:
### Borrowing limit and loan period

A student may borrow **up to 3 books at a time**. The standard loan period is **14 days**. My Loans shows each book’s due date, so check each title separately and return it by its own due date. [1]

### Renewing a book

Request a renewal **before the due date**: open **My Loans**, find the book, and click **Renew**. If nobody else has reserved it, you may renew it **once for 7 extra days**. After renewing, check the new due date in My Loans. If someone has reserved the book, it cannot be renewed and must be returned by its original due date. [1]

### References

- [1] library-guide.pdf

Answer-model activity:
No new answer-model call was recorded; a cached answer may have been used.

Actual system prompt:
No new answer-model call was recorded.

Actual user question:
No new answer-model call was recorded.

Actual conversation history:
No new answer-model call was recorded.


### Repeat a query and inspect its cache

LightRAG first looks for saved **keywords** for the question, mode and language. It still performs retrieval and builds context. It then checks the **answer cache** using the question and query parameters. Reusing an answer skips the final generation call. Query embeddings can still run. Changing `chunk_top_k` can change the answer lookup while allowing keyword reuse.

```mermaid
flowchart LR
    Q["Question"] --> K{"Saved keywords?"}
    K -->|Yes| R["Retrieve evidence and build context"]
    K -->|No| L["Extract and save keywords"]
    L --> R
    R --> A{"Matching saved answer?"}
    A -->|Yes| O["Return saved answer"]
    A -->|No| G["Generate, save and return the answer"]
```

Rerun the preceding answer cell and look for LightRAG’s `Query cache hit` log and the absence of a new answer callback. A saved record alone establishes only that the response was written. The next cell reads an example keyword record and answer record for this question and prints the settings used for the saved answer.


In [10]:
cache_path = storage_dir / "kv_store_llm_response_cache.json"
samples = {}
if cache_path.is_file():
    saved_cache = json.loads(cache_path.read_text(encoding="utf-8"))
    for key, entry in saved_cache.items():
        if key.startswith("mix:") and entry.get("original_prompt") == question:
            samples.setdefault(entry["cache_type"], entry)

keyword_sample = samples.get("keywords")
answer_sample = samples.get("query")

keyword_text = "No keyword entry is saved for this question in mix mode."
if keyword_sample is not None:
    keyword_text = keyword_sample["return"]

if answer_sample is None:
    answer_text = settings_text = "No answer entry is saved for this question in mix mode."
else:
    answer_text = answer_sample["return"]
    settings_text = "\n".join(f"{key} {value}" for key, value in answer_sample["queryparam"].items())

print(f"""
Saved keyword text:
{keyword_text}

Saved answer text:
{answer_text}

Saved query settings:
{settings_text}""")



Saved keyword text:
{"high_level_keywords": ["Student library borrowing policy", "Loan period", "Book renewal"], "low_level_keywords": ["Books", "Student", "Number of books borrowed", "Borrowing duration", "Renewal process"]}

Saved answer text:
### Borrowing limit and loan period

A student may borrow **up to 3 books at a time**. The standard loan period is **14 days**. My Loans shows each book’s due date, so check each title separately and return it by its own due date. [1]

### Renewing a book

Request a renewal **before the due date**: open **My Loans**, find the book, and click **Renew**. If nobody else has reserved it, you may renew it **once for 7 extra days**. After renewing, check the new due date in My Loans. If someone has reserved the book, it cannot be renewed and must be returned by its original due date. [1]

### References

- [1] library-guide.pdf

Saved query settings:
mode mix
response_type Multiple Paragraphs
top_k 40
chunk_top_k 20
max_entity_tokens 6000
max_relati

### Experiment 2 · Does the answer model see the original diagram?

Ask the same question about left-to-right order twice. With `vlm_enhanced=False`, the text model can use the image description written in Lecture 2. With `True`, RAGAnything searches the retrieval prompt for image paths and sends usable images along with text to the vision model. It falls back to text answering if it finds no usable image. Preview-only calls return their requested preview. Keep the image files from Lecture 2 and compare both answers with the diagram on page 2.

The saved comparison in the core workflow reported **0 images sent** for ordinary text answering and **1** for image enhancement. Both answers followed the sequence *check the due date → find the book in My Loans → renew and check the new due date*. In your run, read the count of images in completed vision calls alongside the answers.


In [11]:
image_question = "What steps does the renewal diagram show from left to right?"

text_image_answer = None
text_image_image_count = 0
if run_experiments:
    start = len(calls)
    text_image_answer = await rag.aquery(
        image_question, mode="mix", stream=False, vlm_enhanced=False, enable_rerank=False,
    )
    text_image_image_count = sum(
        call["images"] for call in calls[start:] if call["kind"] == "vision" and "output" in call
    )

print(f"""
Image question:
{image_question}

Answer with vlm_enhanced=False:
{text_image_answer if run_experiments else "Experiment skipped."}

Images sent with vlm_enhanced=False:
{text_image_image_count if run_experiments else "Experiment skipped."}""")


INFO: Executing text query: What steps does the renewal diagram show from left to right?...
INFO: Query mode: mix
INFO: Query nodes: Renewal diagram, Left-to-right order (top_k:40, cosine:0.2)
INFO: Local query: 26 entites, 74 relations
INFO: Query edges: Renewal process, Diagram interpretation, Sequence of steps (top_k:40, cosine:0.2)
INFO: Global query: 23 entites, 40 relations
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 27 entities, 74 relations, 4 vector chunks
INFO: After truncation: 27 entities, 74 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 74 relations
INFO: Round-robin merged chunks: 10 -> 6 (deduplicated 4)
INFO: Final context: 27 entities, 74 relations, 6 chunks
INFO: Final chunks S+F/O: E11/1 E11/2 E8/3 E14/4 E1/5 E7/6
INFO:  == LLM cache == Query cache hit, using cached response as query result
INFO: Text query completed



Image question:
What steps does the renewal diagram show from left to right?

Answer with vlm_enhanced=False:
From left to right, the diagram shows:

1. **Before the due date:** Check the due date.
2. **My Loans:** Find the book.
3. **Renew:** Renew the book and check the new due date. [1]

### References

- [1] library-guide.pdf

Images sent with vlm_enhanced=False:
0


In [12]:
vision_image_answer = None
vision_image_image_count = 0
if run_experiments:
    start = len(calls)
    vision_image_answer = await rag.aquery(
        image_question, mode="mix", stream=False, vlm_enhanced=True, enable_rerank=False,
    )
    vision_image_image_count = sum(
        call["images"] for call in calls[start:] if call["kind"] == "vision" and "output" in call
    )

print(f"""
Answer with vlm_enhanced=True:
{vision_image_answer if run_experiments else "Experiment skipped."}

Images sent with vlm_enhanced=True:
{vision_image_image_count if run_experiments else "Experiment skipped."}""")


INFO: Executing VLM enhanced query: What steps does the renewal diagram show from left to right?...
INFO: Query nodes: Renewal diagram, Left-to-right order (top_k:40, cosine:0.2)
INFO: Local query: 26 entites, 74 relations
INFO: Query edges: Renewal process, Diagram interpretation, Sequence of steps (top_k:40, cosine:0.2)
INFO: Global query: 23 entites, 40 relations
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 27 entities, 74 relations, 4 vector chunks
INFO: After truncation: 27 entities, 74 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 74 relations
INFO: Round-robin merged chunks: 10 -> 6 (deduplicated 4)
INFO: Final context: 27 entities, 74 relations, 6 chunks
INFO: Final chunks S+F/O: E11/1 E11/2 E8/3 E14/4 E1/5 E7/6
INFO: Found 1 image path matches in prompt
INFO: Processed 1 images for VLM
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completi


Answer with vlm_enhanced=True:
From left to right, the diagram shows:

1. **Before the due date:** Check the due date.
2. **My Loans:** Find the book.
3. **Renew:** Check the new due date. [1]

### References

- [1] library-guide.pdf

Images sent with vlm_enhanced=True:
1


A separate interface, `aquery_with_multimodal(query, multimodal_content=...)`, takes an image, table or equation **supplied with the question**. On a cache miss it describes the supplied material, enriches the query and searches the existing index. The experiment above exercises the other path, which reopens an image already indexed from the guide.

**Exit question.** If the answer cites `[1]` but leaves out the reservation condition, which observation would you inspect first? Read the passages returned by `aquery_data` and check whether the condition survived retrieval. Then look for it in the assembled context and compare that context with the final wording.

When you have finished rerunning queries, close the storage handles in this kernel. To query again after closing, restart the kernel and run the lecture from the top. Continue to [Lecture 4 · Recap](lecture_4_recap.ipynb).


In [13]:
await rag.finalize_storages()


INFO: Successfully finalized 12 storages
INFO: Successfully finalized all RAGAnything storages
